# Phase 2: verified splits and honest matching baselines

Phase 2 now covers the full relationship audit, your preserved train/dev/holdout assignments, conservative text normalization, two label-blind matching rules, and development error analysis. Run the cells from top to bottom to read saved results. Expensive rebuilds are disabled by default.

**Important correction:** the original baseline intersected its predictions with the true answers and omitted distractors. That hides false positives. The replacement generates predictions before loading labels, then evaluates every development reference.

In [1]:
from pathlib import Path
import sys, json

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'project_config.json').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from within the project directory.')
sys.path.insert(0, str(ROOT / 'src'))
from business_entity_resolution.workspace import DATASET_DIR
REPORT_PATH = ROOT / 'reports/phase2_relationship_audit.json'
DATABASE_PATH = ROOT / 'data/processed/training_relationships.sqlite'
print('Project:', ROOT)
print('Python:', sys.executable)


Project: /home/a/Projects/Amazon ML hackathon
Python: /home/a/Projects/Amazon ML hackathon/.venv/bin/python


## 1. What are we checking?

A reference and all its true variants should stay in one validation partition. If two references share a target, their labeled groups are connected and should be kept together until that overlap is understood.

| Reference | Targets |
|---|---|
| A | X, Y |
| B | Z |
| C | None |
| D | Y, W |

Here, A and D are connected through Y. C remains a valid singleton group.

The real-data audit checks:
1. Repeated reference IDs in ground truth.
2. Repeated record IDs in the source files.
3. Duplicate targets within a truth list.
4. Targets assigned to several distinct references.
5. Labeled references/targets missing from the sources.
6. Source 1 records missing a ground-truth row.

Shared targets are a grouping finding. Missing IDs or repeated source/reference rows require investigation before splitting.

## 2. How the disk database works

- `reference_rows`: one row for every ground-truth reference, **including singletons**.
- `truth_links`: one row for every labeled reference-target pair.
- `source_records`: record ID, source number, and country from training files; names and addresses are not copied.
- `target_owners`: distinct reference count per labeled target.
- `audit_metadata`: the full report and input SHA-256 hashes.

A batch holds about 10,000 rows before writing. Indexes speed up ID lookup. SQLite uses a bounded page cache and disk-backed temporary storage. Every run builds a new temporary database and replaces the old one only after successful completion; rerunning cannot append accidental duplicate rows.

In [2]:
# Optional rebuild. Leave False unless inputs or audit code changed.
RUN_FULL_AUDIT = False
if RUN_FULL_AUDIT:
    import subprocess
    subprocess.run([sys.executable, str(ROOT / 'scripts/audit_relationships.py')],
                   cwd=ROOT, check=True)
else:
    print('Using saved audit. Full scan intentionally not rerun.')


Using saved audit. Full scan intentionally not rerun.


## 3. Read the full-data results

The JSON is small. Reading it does not load the database into memory. The report records which input files were audited and their checksums; rebuild if the dataset changes.

In [3]:
import pandas as pd
from IPython.display import display

if not REPORT_PATH.is_file():
    raise FileNotFoundError('Run scripts/audit_relationships.py once before viewing results.')
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
print('Audit time (UTC):', report['created_at_utc'])
print('Scope:', report['scope'])
summary_keys = ['reference_rows', 'positive_link_rows', 'singleton_rows',
                'distinct_labeled_targets', 'unmatched_target_record_rows',
                'truth_rows_with_duplicate_targets']
display(pd.DataFrame([{'measure': key, 'count': report[key]} for key in summary_keys]))
display(pd.DataFrame([{'check': key, 'affected_ID_count': value['count']}
                      for key, value in report['checks'].items()]))
print('Structural integrity passed:', report['structural_integrity_passed'])


Audit time (UTC): 2026-09-26T13:53:35.465267+00:00
Scope: Full training ground truth and all three training source ID columns
Structural integrity passed: True


,measure,count
0,reference_rows,2206821
1,positive_link_rows,7638365
2,singleton_rows,123247
3,distinct_labeled_targets,7638365
4,unmatched_target_record_rows,2681854
5,truth_rows_with_duplicate_targets,0


,check,affected_ID_count
0,duplicate_reference_ids,0
1,duplicate_source_ids,0
2,targets_with_multiple_references,0
3,truth_references_missing_from_source1,0
4,source1_records_missing_truth,0
5,truth_targets_missing_from_sources,0


## 4. Interpret the results before writing a split

If each labeled target has exactly one owner, each reference plus its true targets already forms a separate labeled group. There is no need to run a connected-component algorithm on those label edges.

If some targets have multiple owners, inspect those examples and group connected references together. A shared-target count does not by itself tell you how many connected components exist.

Unmatched target records remain useful distractors. Being unmatched to Source 1 does not prove they are mutually distinct businesses.

In [4]:
if not report['structural_integrity_passed']:
    print('STOP: investigate these integrity issues before splitting:')
    print(report['blocking_integrity_issues'])
elif report['grouping_requires_shared_target_components']:
    print('Shared target IDs exist: build connected labeled groups before splitting.')
    display(pd.DataFrame(report['shared_target_examples']))
else:
    print('No target IDs are shared across references.')
    print('Use each Source 1 reference and all its labeled variants as one group.')
    print('A connected-component algorithm is not needed for these label edges.')
print('Next: save and verify group assignments; then build an exact/normalized baseline.')


No target IDs are shared across references.
Use each Source 1 reference and all its labeled variants as one group.
A connected-component algorithm is not needed for these label edges.
Next: save and verify group assignments; then build an exact/normalized baseline.


## 5. See the underlying SQL on a small output

`COUNT(DISTINCT reference_id)` counts separate owners, not repeated copies of the same link. The query returns at most five shared targets. The connection below is read-only; the full database stays on disk.

In [5]:
import sqlite3
if not DATABASE_PATH.is_file():
    raise FileNotFoundError('Local audit database missing. Reports can still be read; rebuild to run SQL.')
connection = sqlite3.connect(DATABASE_PATH.as_uri() + '?mode=ro', uri=True)
try:
    shared = pd.read_sql_query(
        'SELECT target_id, reference_count FROM target_owners '
        'WHERE reference_count > 1 LIMIT 5', connection)
    display(shared)
    # A singleton has a reference row but no truth_links rows.
    singleton = pd.read_sql_query(
        'SELECT reference_id, match_count FROM reference_rows WHERE match_count=0 LIMIT 1', connection)
    display(singleton)
finally:
    connection.close()


,target_id,reference_count


,reference_id,match_count
0,S1-302869473,0


## 6. Limits of the audit

This verifies labeled ID relationships, not perfect real-world identity or complete labels. Separate IDs may represent related businesses across partitions. France remains unseen in labeled training. A local score is not a promised leaderboard score.

## 7. Your saved split is preserved

The original JSON assignments are imported into an indexed SQLite database. All labeled variants follow their Source 1 reference. Unmatched targets also receive fixed assignments, so dev contains realistic distractors. References are stratified by country and number of true matches (0, 1, 2, 3–4, 5–7, 8+).

Labels may define the split and evaluate predictions. They must never filter proposed matches. Holdout membership is structurally verified; holdout performance is not inspected.

In [6]:
RUN_PHASE2 = False  # Keep False to read results without rebuilding the databases.
if RUN_PHASE2:
    import subprocess
    subprocess.run([sys.executable, str(ROOT / 'scripts/run_phase2.py')], cwd=ROOT, check=True)

split_report = json.loads((ROOT / 'reports/phase2_split_summary.json').read_text())
print('Assignments:', split_report['provenance']['mode'])
print('Split ID:', split_report['manifest_id'])
display(pd.DataFrame({
    'references': split_report['reference_counts'],
    'all targets': split_report['target_counts'],
    'unmatched distractors': split_report['distractor_counts'],
}).loc[['train', 'dev', 'holdout']])
display(pd.DataFrame(split_report['strata']).pivot_table(
    index=['country', 'bucket'], columns='split', values='references', aggfunc='sum'))

Assignments: preserved_existing_json
Split ID: 8f33fb5819bd9acf5bb963c278707919ecc4f6b205242bf5ef84f2826caf741e


,references,all targets,unmatched distractors
train,1544769,7223790,1876813
dev,331018,1548174,402537
holdout,331034,1548255,402504


split             dev  holdout   train
country bucket                        
India   0        7402     7404   34545
        1        7120     7121   33227
        2       22489    22490  104948
        3-4     60845    60846  283943
        5-7     33220    33221  155029
        8+       1400     1402    6536
US      0       11084    11085   51727
        1       10753    10754   50182
        2       33792    33794  157699
        3-4     91398    91399  426525
        5-7     49398    49399  230526
        8+       2117     2119    9882

## 8. Conservative normalization

The raw rule requires identical country, name, and address. The normalized rule first applies Unicode NFC, case folding, and whitespace collapse. Both require nonempty country/name/address and retain **all** matching target IDs.

We preserve accents, digits, punctuation, and original strings. Expanding `CT`, `FL`, or `St` without context can change their meaning; that is deferred to separately tested features. Blank addresses are not evidence of equality. This conservative rule will miss many real matches—that is a measurable baseline limitation.

In [7]:
from business_entity_resolution.normalize import normalize_record
samples = [
    {'business_name': '  CAFÉ  Alpha ', 'business_address': '12  CT FL 3', 'country': 'US'},
    {'business_name': 'Cafe\u0301 Alpha', 'business_address': '', 'country': 'US'},
]
display(pd.DataFrame([normalize_record(row['business_name'], row['business_address']) for row in samples]))

,name_raw,name_norm,address_raw,address_norm
0,CAFÉ Alpha,café alpha,12 CT FL 3,12 ct fl 3
1,Café Alpha,café alpha,,


## 9. Honest development results

For each reference, compare sets of predicted and true target IDs:

$$F_{0.5}=\frac{1.25TP}{1.25TP+FP+0.25FN}.$$

Both sets empty earns 1; only one empty earns 0. Average across **all** references. This is not ordinary classification accuracy. Micro precision/recall are useful diagnostics but not the competition metric.

The candidate oracle is a diagnostic upper bound: imagine keeping only true matches from the exact-key candidates. It uses labels only to measure the ceiling, never to generate the baseline predictions.

### Interpretation of this run

Normalized equality recovered **9,655 / 1,145,637 true links (0.8428% recall)**. It produced no observed false positives on dev, but missed **1,135,982 true links**. Its macro F0.5 is **7.3524%**. This is a reproducible starting point, not a competitive matcher.

The candidate oracle equals the actual score. A better classifier restricted to these candidates cannot improve coverage: **we must retrieve more candidates**. This is why Phase 3 comes before logistic regression.

The false negatives below show character substitutions (`Memorial` → `Mem0rial`), word reordering, state abbreviations, domain-style names, and Hindi/English variants. These are illustrative examples, not measured category frequencies.

In [8]:
baseline = json.loads((ROOT / 'reports/phase2_baseline.json').read_text())
assert baseline['manifest_id'] == split_report['manifest_id']
assert baseline['evaluation_partition'] == 'dev' and not baseline['holdout_scored']
metrics = baseline['metrics']
columns = ['macro_f05', 'micro_precision', 'micro_recall', 'tp', 'fp', 'fn',
           'singleton_false_positive_rate', 'candidate_oracle_macro_f05']
display(pd.DataFrame(metrics).T[columns])
print('Predict-nothing macro F0.5:', metrics['raw_exact']['empty_baseline'])
print('Development references:', baseline['reference_count'])
print('All development targets:', baseline['target_count'])
print('Unmatched distractors included:', baseline['distractor_count'])
country_rows = [dict(rule=rule, country=country, **values)
                for rule, m in metrics.items() for country, values in m['by_country'].items()]
display(pd.DataFrame(country_rows)[['rule', 'country', 'references', *columns]])

Predict-nothing macro F0.5: 0.055845905660719355
Development references: 331018
All development targets: 1548174
Unmatched distractors included: 402537


,macro_f05,micro_precision,micro_recall,tp,fp,fn,singleton_false_positive_rate,candidate_oracle_macro_f05
raw_exact,0.05585,1.0,0.000002,2,0,1145635,0.0,0.05585
normalized_exact,0.073524,1.0,0.008428,9655,0,1135982,0.0,0.073524


,rule,country,references,macro_f05,micro_precision,micro_recall,tp,fp,fn,singleton_false_positive_rate,candidate_oracle_macro_f05
0,raw_exact,India,132476,0.055883,1.0,0.000004,2,0,458934,0.0,0.055883
1,raw_exact,US,198542,0.055827,NaN,0.000000,0,0,686701,0.0,0.055827
2,normalized_exact,India,132476,0.065754,1.0,0.004711,2162,0,456774,0.0,0.065754
3,normalized_exact,US,198542,0.078709,1.0,0.010912,7493,0,679208,0.0,0.078709


## 10. Inspect the errors

A false positive is a proposed ID absent from the supplied truth. A false negative is a supplied true ID the rule missed. These examples are deterministic first-ID examples, not a representative random sample. Labels can be incomplete; do not relabel rows simply because they look similar.

Look for missing addresses, formatting differences, abbreviations, reordered words, and altered names. Phase 3 should retrieve candidates despite those differences, while keeping other businesses and distractors in the search pool.

In [9]:
errors = json.loads((ROOT / 'reports/phase2_error_examples.json').read_text())
for kind, examples in errors['normalized_exact'].items():
    print(kind)
    display(pd.DataFrame(examples))
run = json.loads((ROOT / 'reports/phase2_run.json').read_text())
print('Complete run seconds:', run['elapsed_seconds'])
print('Peak process memory (MiB):', run['peak_process_ram_mib'])

false_positive
false_negative
Complete run seconds: 392.89
Peak process memory (MiB): 973.11


""


,reference_id,country,reference_name,reference_address,target_id,target_name,target_address
0,S1-100005592,US,Memorial Ministries,"105 Canal Street, Lagrange, IN",S3-298224934,Mem0rial Ministries Co,"105 Canal Street, Lagrange, Indiana"
1,S1-100005592,US,Memorial Ministries,"105 Canal Street, Lagrange, IN",S3-640765651,Memorial Memorial Ministries,"Lagrange, Indiana, 105 Canal Street"
2,S1-100010277,US,Concordia's Complete Apparel,"2712 Camille Street, Pasadena, TX",S2-247097312,Concordia's Apparel Complete,"2712- CAMILLE STREET, PASADENA, TX"
3,S1-100010277,US,Concordia's Complete Apparel,"2712 Camille Street, Pasadena, TX",S2-606992934,*** Concordia's Complete Apparel Ltd,"2712- CAMILLE ST, PASADENA, TX"
4,S1-100010277,US,Concordia's Complete Apparel,"2712 Camille Street, Pasadena, TX",S3-960005804,Concordia'S Complete-Apparel Complete-Apparel,"2712 Camille Street, Pasadena, Texas"
5,S1-100012185,US,Interstate School of Medicine,"9925 Great Oaks Way, Fairfax City, VA",S3-535035678,Interstate School,"9925 Great Oaks Way, Fairfax City, Virginia"
6,S1-100012185,US,Interstate School of Medicine,"9925 Great Oaks Way, Fairfax City, VA",S3-576158732,interstateschoolmedicine.com,"9925 Great Oaks Way, Fairfax City, Virginia"
7,S1-100012185,US,Interstate School of Medicine,"9925 Great Oaks Way, Fairfax City, VA",S3-91756946,Interstateschoolmedicine.Com,"9925 Great Oaks Way, Virginia, Fairfax City"
8,S1-100017441,India,Great Agro Private Limited,"15/49 S/F, Subhash Nagar, New Delhi, Delhi",S2-974991352,ग्रेट एग्रो प्राइवेट लिमिटेड,"75/49 S/F, SUBHASH NAGAR, NEW DELHI, Delhi"
9,S1-100017441,India,Great Agro Private Limited,"15/49 S/F, Subhash Nagar, New Delhi, Delhi",S3-791599964,GA,"15/49 S/F, Subhash Nagar, New Delhi, Delhi"


## 11. Phase 2 completion and next step

Read `reports/phase2_completion.md` for the reviewed findings, measurements, and remaining limits. The two complete dev TSVs live in `data/processed/phase2/`; **they are not test submissions**. Databases and predictions are excluded from Git. Small reports and source code can be shared.

Phase 3 begins with candidate retrieval, not a more complicated classifier. A classifier cannot recover an entity that retrieval never proposes. Benchmark complementary name/address routes on dev, then measure candidate recall, oracle macro F0.5, candidates per reference, time, and RAM. Train/tune future models on the appropriate partitions; keep holdout locked until a genuine confirmation checkpoint.